# Notebook 00 — Setup and your first local AI call

**Time:** about 30 minutes (plus the one-time model download)

**What you will learn**
- How to start Ollama and download a free model on Windows 11
- How to send a question to the model from Python
- What is inside a model's reply (text, tokens, time)
- How a *system message* changes the model's behaviour

**Everything runs on your own computer.** No API key, no account, no internet after the model is downloaded.

## Part 1 — One-time setup on Windows 11

**Easiest:** double-click `setup_windows.bat` in this folder once. Later, double-click `start_workshop.bat`.

Or type the commands yourself in **PowerShell**, one at a time:

```powershell
winget install -e --id Ollama.Ollama          # install Ollama (it then runs in the background)
winget install -e --id Python.Python.3.12     # install Python
```

**Close PowerShell and open it again** (so Windows finds the new programs), then:

```powershell
ollama pull llama3.2                          # download the model (about 2 GB, needs internet once)
ollama list                                   # llama3.2 should appear in the list
```

Then set up Python for the workshop (inside the workshop folder):

```powershell
py -m venv .venv                              # a private Python environment for this workshop
.venv\Scripts\Activate.ps1                    # if Windows blocks this, see the table at the end
python -m pip install -r requirements.txt
jupyter lab                                   # opens JupyterLab in your browser
```

💡 Every time you come back later, just double-click `start_workshop.bat`.

## Part 2 — Check that everything works

In a notebook, click a code cell and press **Shift + Enter** to run it. Always run cells from top to bottom.

In [ ]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

In [ ]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

## Part 3 — Your first question

`ask()` sends your text to the model and returns its answer. The first call can take 10–60 seconds,
because Ollama loads the model into memory. Later calls are faster.

In [ ]:
answer = ask("What is an AI agent? Answer in two sentences.")
print(answer)

## Part 4 — What is inside a reply?

`ask()` only gives back the text. Here we call `ollama.chat()` directly to see the whole reply.

- **Tokens** are small pieces of words. The model reads *input tokens* and writes *output tokens*.
- More tokens = more work for your computer = more time.

In [ ]:
message = {"role": "user", "content": "Say hello in five words."}
reply = ollama.chat(model=MODEL, messages=[message])
seconds = reply.total_duration / 1e9     # nanoseconds → seconds

print("Answer        :", reply.message.content)
print("Input tokens  :", reply.prompt_eval_count)
print("Output tokens :", reply.eval_count)
print("Total time    :", round(seconds, 1), "seconds")

## Part 5 — The system message sets the model's role

The **system message** tells the model *how* to behave. The **user message** is the actual question.
Same question, three different system messages:

In [ ]:
question = "What is an AI agent?"

roles = [
    "Explain to a 10-year-old child in one short sentence.",
    "You are a poet. Answer with a 3-line poem.",
    "You are an engineer. Answer in one technical sentence.",
]

for role in roles:
    print("ROLE:", role)
    print("ANSWER:", ask(question, system=role))
    print("-" * 60)

## ✍️ Try it yourself

1. Change `my_question` below to anything you like and run the cell.
2. Change `my_role` to give the model a different personality.

In [ ]:
my_question = "Give me three tips for learning Python."
my_role = "You are a friendly teacher. Use bullet points."

print(ask(my_question, system=my_role))

## 🧰 If something goes wrong

| Message | What to do |
|---|---|
| `Ollama is not running` | Open **Ollama** from the Start menu (or run `ollama serve` in PowerShell) |
| `llama3.2 is missing` | In PowerShell: `ollama pull llama3.2` |
| `ModuleNotFoundError: ollama` | Start JupyterLab with `start_workshop.bat`, or run `setup_windows.bat` again, then restart the kernel |
| `running scripts is disabled` (PowerShell) | Run once: `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`, or just use the `.bat` files |
| `ollama` or `py` is not recognized | Close PowerShell and open it again (or restart Windows) |
| Answers are very slow | Normal on a computer without a GPU. Try the smaller model: `ollama pull llama3.2:1b` and set `MODEL = "llama3.2:1b"` |
| Strange errors after editing cells | Menu **Kernel → Restart Kernel and Run All Cells** |

## ✅ Summary
- Ollama runs the model on **your** computer; Python talks to it with `ollama.chat()`.
- `ask(prompt, system)` is our small helper: system message = role, user message = question.
- Replies tell us the number of tokens and the time taken.

**Next:** Notebook 01 — making the model work in several steps (an *agentic workflow*).